# MMTD Song Recommender

Refactored from the original Colab TCC notebook. Logic lives in the `mmtd_recommender` package (`pip install -e .`); this notebook is just a thin demo/report layer.

Two models:
- **Popularity** — top tracks by tweet count (cold-start fallback for new users).
- **Item-based collaborative filtering** — cosine similarity over a sparse track×user interaction matrix, recommends tracks similar to what the user already tweeted about.

In [1]:
from pathlib import Path

from mmtd_recommender import (
    ItemBasedCFRecommender,
    PopularityRecommender,
    download_dataset,
    load_artists,
    load_tracks,
    load_tweets,
    tracks_for_user,
)

DATA_DIR = Path("data/raw")

## Load the dataset

Downloads once into `data/raw/` (skipped on subsequent runs). `load_artists` handles the one malformed row in `artists.txt` (an artist name with a literal embedded tab) that broke the original `pd.read_csv` call.

In [2]:
paths = download_dataset(DATA_DIR)

artists = load_artists(paths["artists.txt"])
tracks = load_tracks(paths["track.txt"])
tweets = load_tweets(paths["tweet.txt"])

print(f"artists: {len(artists):,}")
print(f"tracks:  {len(tracks):,}")
print(f"tweets:  {len(tweets):,}")

artists: 523,479
tracks:  134,199
tweets:  1,090,726


## Explore the dataset

In [3]:
print(f"unique users:  {tweets['tweet_userId'].nunique():,}")
print(f"unique tracks: {tweets['tweet_trackId'].nunique():,}")

unique users:  215,375
unique tracks: 134,199


## Popularity model

Top 5 most-tweeted songs.

In [4]:
pop_model = PopularityRecommender(tweets, tracks, artists)
%time pop_model.top_songs(5)

CPU times: total: 93.8 ms
Wall time: 89.6 ms


,track_id,score,track_title,artist_name
0,141574,4387,Someone Like You,Adele
1,1479214,3331,Paradise,Coldplay
2,2966419,2742,Somebody That I Used to Know,Gotye
3,141567,2543,Set Fire to the Rain,Adele
4,4098232,2532,The One That Got Away,Katy Perry


## Item-based collaborative filtering

Pick a user, look at what they've already tweeted about, and recommend similar tracks. New users (no tweet history) fall back to the popularity model.

In [5]:
# example users
#   265101134  (14 tweets)
#   58937384   (854 tweets)
#   92235951   (75 tweets)
#   250253081  (2 tweets)
#
# new user (no history)
#   43254

test_user = 265101134

user_tracks = tracks_for_user(tweets, test_user)
print(f"user {test_user} has tweeted about {len(user_tracks)} distinct tracks")

user 265101134 has tweeted about 11 distinct tracks


In [6]:
if not user_tracks:
    print("-- new user: falling back to popularity recommendation --")
    recommendation = pop_model.top_songs(topn=10)
else:
    cf_model = ItemBasedCFRecommender(tweets, tracks, artists)
    print("-- item-based CF recommendation --")
    %time recommendation = cf_model.recommend(user_tracks, topn=10)

recommendation

-- item-based CF recommendation --
CPU times: total: 46.9 ms
Wall time: 41.6 ms


,track_id,score,track_title,artist_name
0,2106305,0.487080,Lust for Life,Drake
1,2106349,0.481039,Over My Dead Body,Drake
2,2106383,0.459261,Shot for Me,Drake
3,2106361,0.454158,Practice,Drake
4,7069026,0.424264,I Don't Give a Fuk,T Pain
5,28731,0.424264,Words to My First Born,2Pac
6,10557906,0.424264,Save You,One Chance
7,909493,0.424264,I Miss You,Blood Raw
8,2704794,0.424264,Friends B-4 Lovers,Full Force
9,7899102,0.424264,I Cry,Trick Daddy
